# Claude Developer Platform — Build-Along

**Level:** 300 (Intermediate) | **Model:** Claude Sonnet 5 (`claude-sonnet-5`)

## What We're Building

A multi-tool **support ticket agent** that reads ticket details, searches a knowledge base, and produces a structured resolution — using the Claude API directly, no framework.

**Customer scenario:** TechFlow (mid-market B2B SaaS, 500+ tickets/day) wants to automate Tier 1 triage so human agents can focus on complex escalations.

## Learning Objectives

1. Implement a multi-tool agentic loop (`while stop_reason == "tool_use"`)
2. Compose structured outputs with tool use in a single agent
3. Integrate adaptive thinking with `output_config.effort` to control reasoning depth
4. Stream thinking, tool calls, and responses in real-time

## Where Does This Fit?

There are several ways to build with Claude. This session focuses on the **Messages API** — the lowest level, giving you full control over the request/response cycle:

| Surface | What It Is | When to Use It |
|---------|-----------|----------------|
| **Messages API** ← *this session* | Direct HTTP/SDK calls to Claude | Full control over agentic loops, custom orchestration, production backends |
| **Agent SDK** | Python framework with built-in tool dispatch | Rapid agent prototyping, when you want orchestration handled for you |
| **Claude Code** | CLI-based coding agent | Developer productivity, repo-level tasks, interactive coding |
| **claude.ai / Claude for Enterprise** | Chat interface with Projects, MCP | End-user workflows, enterprise knowledge work, non-developer use cases |

Today you build the raw loop. Understanding this makes everything above it clearer.

---

**Prepared for Partner Basecamp participants.** Not for reproduction or redistribution as training material — you're free to apply these patterns in your own client work.

## Setup

Run the cells below to install dependencies, configure your API key, and load mock data.

- **Your API key** goes in a `.env` file the setup cell creates — paste it there once (it's gitignored and survives kernel restarts).
- **Look for ✏️ YOUR TURN cells** — they mark exactly where you write code today.
- **API cells take time** — an agent run with thinking can take 30–90 seconds. The `[*]` next to a cell means it's still working.


### Setup — connect to Claude

Run the next cell first. The setup cell creates a **`.env` file** the first time you run it (gitignored — your key is never committed). Open it, paste your key after `ANTHROPIC_API_KEY=`, save, and re-run — it survives kernel restarts, so you paste once. *(No `.env` yet? A hidden input box appears as a fallback.)* You're locked in when you see the green **"✓ API key verified"** banner. Red banner? Do what it says and run the cell again.

In [1]:
# ── Install & Import ──
# Install dependencies into THIS kernel — safe to re-run; survives locked-down (PEP 668) Pythons.
import importlib.util, os, subprocess, sys

# ── Environment guard ─────────────────────────────────────────────────────
# Lives in the SAME cell as the installer below so it can't be skipped: no
# package is ever installed into a bare system Python (the old worst case was
# --break-system-packages against an IT-managed machine). If this stops you,
# see SETUP.md → "Why the notebook just stopped".
def _in_isolated_env():
    """True when the RUNNING KERNEL is a venv/virtualenv/conda env or Colab.
    Judged from the interpreter itself (sys.*). Activation env vars inherited
    from the launching shell are trusted only when sys.executable actually
    lives inside the environment they point to — a system-Python kernel
    launched from an activated terminal still inherits VIRTUAL_ENV and must
    NOT pass."""
    if "google.colab" in sys.modules:
        return True  # Colab sandboxes its own disposable runtime
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405 venv (python -m venv); also most conda envs
    if hasattr(sys, "real_prefix"):
        return True  # legacy virtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # hearsay from the shell — the kernel lives elsewhere
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # conda's shared `base` doesn't count as isolated
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  Environment guard bypassed (BASECAMP_ALLOW_SYSTEM_PYTHON=1) — "
          "installing into this Python on purpose.")
elif not _in_isolated_env():
    _HEAD = "✗ System Python detected — stopped before installing anything"
    _BODY = (
        "Installing packages here changes Python for your whole machine — on a\n"
        "corporate-managed laptop that can mean an IT ticket.\n"
        "\n"
        "Fix (2 steps):\n"
        "  1. In a terminal, from the repo root:\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. In VS Code: click the kernel name (top-right) → Select Another Kernel →\n"
        "     Python Environments → pick the one ending in .venv → run this cell again.\n"
        "\n"
        "Using conda? `conda activate <env>` (not `base`), then pick that kernel.\n"
        "Facilitator on a self-managed machine? BASECAMP_ALLOW_SYSTEM_PYTHON=1 bypasses."
    )
    _shown = False
    try:  # same banner treatment as the API-key check below (green box, red variant)
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "Environment guard stopped this cell — see the message above."
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ Environment guard: isolated interpreter detected, safe to install")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: list of (import_name, pip_spec). Install only what is missing,
    into the running interpreter. Tries a normal install, then user-space, then a
    PEP 668 override (user-space first, system-wide only as a last resort). Every
    attempt is silent — pip's output is captured, not streamed — so a locked-down
    Python (Homebrew or Debian, PEP 668) no longer dumps a scary
    'externally-managed-environment' wall of text when a fallback is what actually
    succeeds. Only if every strategy fails does it surface the reason, with the
    venv fix instead of a raw traceback."""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "(no output from pip)"
    raise SystemExit(
        "\n  Couldn't install: " + ", ".join(missing) + "\n"
        "  This Python is locked down (PEP 668) or offline. Quickest fix is a venv:\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows: see SETUP.md\n"
        f"      pip install {' '.join(missing)}\n"
        "  Then pick the .venv interpreter in VS Code (kernel picker, top-right) and Run All.\n"
        "  Corporate proxy or PyPI blocked? See SETUP.md in the repo root.\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ Dependencies ready")

import anthropic
import json
import time
import os

# ── API Key Configuration ──
import os

def _status(ok, msg):
    """Green/red banner in notebooks; plain text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Connect to Claude — Anthropic API or Amazon Bedrock ──
# Works with either credential type; the cell figures out which you have.
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  plus  AWS_REGION=us-east-1
# Put whichever you use in the .env file this cell creates (gitignored — never committed),
# or export it in your shell. A value in the shell wins over the .env file.
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """Nearest existing .env walking up from the working dir (so one root .env serves every
    exercise); if none exists yet, point at the repo root — or this folder if the notebook
    was opened on its own."""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# Tiny .env parser (no python-dotenv dependency). Re-read on every run, so pasting your
# key and re-running picks it up. A real value in the environment (shell / Claude Code / CI)
# wins; the placeholder never sticks.
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """Warning-yellow banner + stop, so setup fails here rather than several cells later."""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("Credentials missing — see the message above.")


if PROVIDER is None:
    _needs_credentials(
        "📋 Add your credentials to continue",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Using the Anthropic API? Set:\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Using Amazon Bedrock? Set both:\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # the region your models are enabled in\n"
        "\n"
        "Save the file, then click ▶ on this cell again."
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrock needs a region",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "and add the region your Bedrock models are enabled in, e.g.:\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "Save the file, then click ▶ on this cell again."
    )


def _model(name):
    """Bedrock model IDs carry an `anthropic.` prefix; the Anthropic API uses the bare ID."""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# Named models the exercise uses — resolved for whichever provider you're on.
MODEL = _model("claude-sonnet-5")        # the workhorse for this exercise
FAST_MODEL = _model("claude-haiku-4-5")  # cheap + quick (connection check, judges)
BIG_MODEL = _model("claude-opus-4-8")    # when you want to try a larger model


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# Connection check — verifies the credential AND that this model is reachable for you.
# On Bedrock a valid key can still 404 if the model isn't enabled in your account/region,
# so we ping the real model ID rather than just checking the credential's shape.
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("Model not available — see the message above.")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "That Bedrock key was rejected. Check AWS_BEARER_TOKEN_BEDROCK and that "
                       "it has Bedrock invoke permissions, then run this cell again.")
    else:
        _status(False, "That key was rejected. Run this cell again and paste the whole key "
                       "(it starts with sk-ant-).")
    raise SystemExit("Credentials not accepted - re-run this cell and try again.")
except Exception as exc:
    _status(False, "Could not reach the API (" + type(exc).__name__ + "). Check your "
                   "connection, then run this cell again.")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # later cells / !python pick it up
        _status(True, "API key verified - you're connected to Claude.")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# The working client. Longer timeout: needed for max_tokens>21333 with non-streaming calls.
client = _make_client(timeout=900.0)


⚠️  Environment guard bypassed (BASECAMP_ALLOW_SYSTEM_PYTHON=1) — installing into this Python on purpose.
✓ Dependencies ready


In [2]:
# ── Sample Ticket Data ──

TICKETS = {
    "TKT-1042": {
        "id": "TKT-1042", "customer": "Acme Corp", "priority": "high",
        "product_area": "billing",
        "description": "We were charged twice for our March invoice. Invoice #INV-2024-0342 shows $4,500 but our bank shows two identical charges on March 3rd. Need immediate refund of the duplicate charge.",
        "status": "open"
    },
    "TKT-1043": {
        "id": "TKT-1043", "customer": "DataFlow Inc", "priority": "medium",
        "product_area": "api",
        "description": "Our webhook endpoint stopped receiving events after we rotated API keys yesterday. We've verified the new key works for REST calls but webhooks are still failing. Getting 401 errors in the webhook logs.",
        "status": "open"
    },
    "TKT-1044": {
        "id": "TKT-1044", "customer": "CloudScale Ltd", "priority": "low",
        "product_area": "feature_request",
        "description": "Would love to see bulk export functionality in the dashboard. Currently we have to export reports one at a time which is painful when we need quarterly summaries across 50+ projects.",
        "status": "open"
    },
    "TKT-1045": {
        "id": "TKT-1045", "customer": "SecureNet Systems", "priority": "critical",
        "product_area": "account",
        "description": "Our admin account (admin@securenet.io) is locked out after failed MFA attempts. We have 47 team members who can't access the platform because SSO is tied to this admin account. This is blocking all work.",
        "status": "open"
    },
    "TKT-1046": {
        "id": "TKT-1046", "customer": "MedTech Solutions", "priority": "high",
        "product_area": "api",
        "description": "Our production integration started returning intermittent 500 errors around 2am last night. About 15% of API calls are failing. We haven't changed anything on our end. Errors seem random - sometimes the same request works on retry. Our team in Singapore is blocked and we need this resolved ASAP.",
        "status": "open"
    },
}

KB_ARTICLES = {
    "KB-001": {"title": "Processing Duplicate Payment Refunds", "content": "For duplicate charges: 1) Verify the duplicate in the billing system, 2) Issue refund through the payment processor (takes 3-5 business days), 3) Send confirmation email with refund reference number. Escalate if amount exceeds $10,000."},
    "KB-002": {"title": "Webhook Authentication After Key Rotation", "content": "When API keys are rotated, webhook signing secrets must also be updated. Go to Settings > Webhooks > Edit endpoint, and regenerate the signing secret. The old secret is invalidated immediately on key rotation. Common mistake: rotating the API key but not the webhook signing secret."},
    "KB-003": {"title": "Bulk Export Feature (Roadmap)", "content": "Bulk export is on the Q3 roadmap. Workaround: Use the REST API's /reports/export endpoint with date range parameters to programmatically export multiple reports. See API docs for batch export examples."},
    "KB-004": {"title": "Admin Account Lockout Recovery", "content": "For locked admin accounts: 1) Verify identity through the secondary email on file, 2) Reset MFA through the admin recovery flow at /admin/recover, 3) Temporary access can be granted through support-level override (requires manager approval). Critical: If SSO is blocked, enable the bypass login at /login/direct for affected users."},
    "KB-005": {"title": "API Rate Limiting Best Practices", "content": "Default rate limits: 100 requests/minute for standard plans, 1000/minute for enterprise. Use exponential backoff with jitter for retries. Monitor usage via the X-RateLimit headers in responses."},
    "KB-006": {"title": "Invoice Discrepancy Resolution", "content": "For billing discrepancies: Check the billing audit log for the account, compare with payment processor records, and verify no pending transactions. Contact finance team for adjustments over $5,000."},
    "KB-007": {"title": "Intermittent 500 Errors Troubleshooting", "content": "For intermittent server errors: 1) Check the status page for known outages, 2) Review rate limit headers - 429s can masquerade as 500s behind load balancers, 3) Check if errors correlate with payload size or specific endpoints, 4) Enable request ID logging and contact support with specific request IDs for investigation. If >10% error rate persists for >1 hour, escalate to engineering."},
}

def get_ticket(ticket_id: str) -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        return json.dumps(ticket)
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

def search_kb(query: str) -> str:
    query_lower = query.lower()
    results = []
    for article_id, article in KB_ARTICLES.items():
        if any(word in article["title"].lower() or word in article["content"].lower()
               for word in query_lower.split() if len(word) > 2):
            results.append({"id": article_id, **article})
    if not results:
        results = [{"id": "KB-000", "title": "No matches found", "content": "No relevant articles found. Consider escalating to Tier 2 support."}]
    return json.dumps(results[:3])

def resolve_ticket(ticket_id: str, resolution: str, status: str = "resolved") -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        ticket["status"] = status
        ticket["resolution"] = resolution
        return json.dumps({"success": True, "ticket_id": ticket_id, "new_status": status})
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

TOOL_FUNCTIONS = {"get_ticket": get_ticket, "search_kb": search_kb, "resolve_ticket": resolve_ticket}

def execute_tool(name: str, input_data: dict) -> str:
    func = TOOL_FUNCTIONS.get(name)
    if func:
        return func(**input_data)
    return json.dumps({"error": f"Unknown tool: {name}"})

print("Mock tools and sample data loaded!")
print(f"   Available tickets: {', '.join(TICKETS.keys())}")
print(f"   Knowledge base articles: {len(KB_ARTICLES)}")

Mock tools and sample data loaded!
   Available tickets: TKT-1042, TKT-1043, TKT-1044, TKT-1045, TKT-1046
   Knowledge base articles: 7


---
# Part 1: Multi-Tool Agentic Loop

TechFlow's Tier 1 support team currently handles each ticket manually: look up the customer, search the knowledge base, categorize the issue, draft a resolution. That's 500+ tickets per day, ~8 minutes each. They want Claude to do this autonomously.

We'll build the agent that replaces that workflow — three tools, one loop, no framework.

> **Key concepts:** Claude Sonnet 5 supports *adaptive thinking* — it automatically decides how much to reason based on task complexity. We enable it with `thinking={"type": "adaptive"}` on every API call.

## Part 2: Define Tool Schemas

TechFlow's support agent needs access to three systems: the ticketing platform (to look up details), the knowledge base (to find solutions), and the resolution engine (to close tickets). Each of these becomes a tool schema that tells Claude what's available and how to call it.

The `description` field is critical — it's how Claude decides *which* tool to use for a given step. Vague descriptions lead to wrong tool selection.

> 📖 **Reference:** [Tool use documentation](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)

### ✏️ YOUR TURN — complete the `search_kb` and `resolve_ticket` tool schemas

`get_ticket` is already filled in as your reference. Complete the empty `description` strings and the `status` enum wherever you see `<-- fill this in`.

In [3]:
# TODO: Define tool schemas for search_kb and resolve_ticket
# Each tool needs: name, description, input_schema (with properties and required)

tools = [
    # ✅ Example: get_ticket is done — use this as your reference for the next two
    {
        "name": "get_ticket",
        "description": "Retrieve full details for a support ticket by its ID, including customer, priority, product area, and description.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string", "description": "The ticket ID, e.g. TKT-1042"}
            },
            "required": ["ticket_id"]
        }
    },

    # ✏️ YOUR TURN: fill in search_kb
    # Hint: the description is how Claude decides *when* to call this tool — make it specific
    {
        "name": "search_kb",
        "description": "Search the knowledge base for articles relevant to a support issue. Use this after looking up the ticket, before resolving it, to find documented solutions, troubleshooting steps, or procedures that apply to the customer's problem.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Keywords describing the issue or product area to search for, e.g. 'duplicate charge refund' or 'webhook 401 error'"}
            },
            "required": ["query"]
        }
    },

    # ✏️ YOUR TURN: fill in resolve_ticket
    # Hint: status should be an enum — what are the three possible resolution states?
    {
        "name": "resolve_ticket",
        "description": "Close out a support ticket with a written resolution. Call this after investigating the ticket and searching the knowledge base, once you have a clear diagnosis and next steps to give the customer.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"},
                "resolution": {"type": "string"},
                "status": {"type": "string", "enum": ["resolved", "escalated", "pending"]}
            },
            "required": ["ticket_id", "resolution", "status"]
        }
    }
]

print(f"Defined {len(tools)} tool schemas: {[t['name'] for t in tools]}")

Defined 3 tool schemas: ['get_ticket', 'search_kb', 'resolve_ticket']


## Part 3: Build the Agentic Loop

Here's the core automation: when a ticket comes in, Claude should look it up, search for a solution, and resolve it — all without human intervention. The agentic loop makes this possible: `while response.stop_reason == "tool_use"`, extract tool calls, execute them, append results, and call the API again. Claude decides the sequence; your code just orchestrates.

**Key:** Pass `response.content` back as-is — it may contain thinking blocks alongside tool_use blocks. Claude Sonnet 5 uses `thinking={"type": "adaptive"}` on every call.

> 📖 **Reference:** [Agentic tool use patterns](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/agentic-tool-use)

### ✏️ YOUR TURN — build the agentic loop

Fill in every `___` blank in `run_agent()` below: the loop condition, the tool-result wiring, and the follow-up API call.

In [4]:
SYSTEM_PROMPT = """You are a Tier 1 support agent for TechFlow, a B2B SaaS platform that provides project management and team collaboration tools to mid-market companies.

## Your Role
You handle incoming support tickets by investigating issues, finding solutions in the knowledge base, and resolving tickets with clear, actionable guidance.

## Process
1. ALWAYS look up the ticket first to understand the full context
2. Search the knowledge base for relevant solutions and procedures
3. Resolve the ticket with a detailed resolution that includes specific next steps

## Guidelines
- Be thorough: always search the KB before resolving, even if the issue seems straightforward
- Be specific: include exact steps, links, and timeframes in resolutions
- Escalate when needed: if confidence is low or the issue requires privileged access, mark for escalation
- Categorize accurately: billing, technical, account, or feature_request

## Escalation Criteria
- Financial issues over $10,000
- Security-related account compromises
- Issues requiring engineering intervention
- Customers with Enterprise SLA (response within 1 hour)

## TechFlow Product Tiers
- Starter ($29/user/month): Basic project management, 5GB storage, email support, 5 projects max, community forums
- Professional ($79/user/month): Advanced analytics, 100GB storage, priority support, API access, unlimited projects, custom fields, Gantt charts, time tracking
- Enterprise (custom pricing): SSO/SAML, unlimited storage, dedicated CSM, custom integrations, SLA guarantees, audit logs, advanced security, custom branding, priority API rate limits

## Common Issue Categories and Routing
- Billing: Invoice discrepancies, payment failures, plan changes, refund requests, subscription cancellations, proration questions
- Technical: API errors, integration issues, webhook failures, performance problems, data export issues, browser compatibility
- Account: Login issues, MFA problems, SSO configuration, permission changes, team management, user provisioning
- Feature Requests: Product feedback, roadmap inquiries, workaround requests, beta access requests

## Response Templates
When resolving billing issues, always include: transaction ID, refund timeline, and confirmation email details.
When resolving technical issues, always include: steps to reproduce, workaround if available, and engineering ticket number if escalated.
When resolving account issues, always include: security verification steps taken and any temporary access granted.

## SLA Requirements
- Starter: 24-hour response time, business hours only
- Professional: 4-hour response time, extended hours (6am-10pm)
- Enterprise: 1-hour response time, 24/7 support, dedicated Slack channel

## Tone
Professional, empathetic, and solution-oriented. Acknowledge the customer frustration before jumping to the solution. Use the customer name when available. Reference the specific product tier for relevant guidance."""


# TODO: Implement run_agent(user_message)
# 1. Create messages list with the user message
# 2. Call client.messages.create() with:
#    - model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT, tools=tools
#    - thinking={"type": "adaptive"}
#    - messages=messages
# 3. While response.stop_reason == "tool_use":
#    a. Loop through response.content, find tool_use blocks
#    b. Execute each tool with execute_tool(block.name, block.input)
#    c. Build tool_result dicts with tool_use_id and content
#    d. Append assistant response + tool results to messages
#       (pass ALL content blocks back, including thinking blocks!)
#    e. Call the API again
# 4. Return the final response

def run_agent(user_message: str):
    """Run the support ticket agent."""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL,
        max_tokens=32000,
        system=SYSTEM_PROMPT,
        tools=tools,
        thinking={"type": "adaptive"},
        messages=messages
    )

    # ✏️ YOUR TURN: fill in every ___ blank below (loop while Claude still wants to use tools)
    while response.stop_reason == "tool_use":  # <-- what stop_reason means "Claude wants to call a tool"?

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,  # <-- which field on `block` links this result back to the tool call?
                    "content": str(result)
                })

        # TODO: Append assistant turn + tool results back to messages
        # Key: pass ALL content blocks (including any thinking blocks) back — not just text
        messages.append({"role": "assistant", "content": response.content})  # <-- what goes here?
        messages.append({"role": "user", "content": tool_results})

        # TODO: Call the API again with the updated messages
        response = client.messages.create(
            model=MODEL,
            max_tokens=32000,
            system=SYSTEM_PROMPT,
            tools=tools,
            thinking={"type": "adaptive"},
            messages=messages
        )

    return response


# Test it!
# response = run_agent("Resolve ticket TKT-1042")
# for block in response.content:
#     if block.type == "text" and block.text.strip():
#         print(f"\n Final response:\n{block.text}")

## Part 4: Add Structured Output

TechFlow's downstream systems need machine-readable resolutions — the ticketing platform updates its database, the analytics dashboard tracks categories, and the escalation router checks the `escalation_needed` flag. Free-text responses won't cut it.

`output_config.format` constrains Claude's text response to match a JSON schema. **Important:** The format constraint applies to *all* text output, so we only add it on the final API call — after the tool loop completes. During the loop, Claude uses tools normally without format constraints. Once tools are done, we make one more call with `output_config.format` and `tool_choice={"type": "none"}` to get a structured JSON resolution.

**Note:** With adaptive thinking, the response may contain `[thinking, text]` blocks. The structured JSON is in the *last* text block.

> 📖 **Reference:** [Structured outputs / JSON mode](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)

### ✏️ YOUR TURN — finish `run_agent_structured()`

The tool loop is already written for you. Fill in the two `___` blanks on the final call: `output_config` and `tool_choice`.

In [ ]:
# ✅ RESOLUTION_SCHEMA is already defined below — read it, then implement run_agent_structured()
RESOLUTION_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "diagnosis": {"type": "string", "description": "Root cause analysis of the issue"},
            "solution_steps": {"type": "array", "items": {"type": "string"}, "description": "Ordered steps to resolve"},
            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
            "escalation_needed": {"type": "boolean"},
            "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}
        },
        "required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"],
        "additionalProperties": False
    }
}


def get_structured_result(response) -> dict:
    """Extract the structured JSON from the last text block in the response."""
    # With adaptive thinking, content may be [thinking, text] - JSON is in the last text block
    text_blocks = [b for b in response.content if b.type == "text" and b.text.strip()]
    if text_blocks:
        return json.loads(text_blocks[-1].text)
    return None


def run_agent_structured(user_message: str) -> dict:
    """Run the agent with structured JSON output."""
    # Step 1: Run the tool loop — same as run_agent(), NO output_config here
    # (format constrains ALL text output, so tools won't work with it active)
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(
        model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
        tools=tools, thinking={"type": "adaptive"}, messages=messages
    )
    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(
            model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
            tools=tools, thinking={"type": "adaptive"}, messages=messages
        )

    # Step 2: One final call to get structured output
    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})
    # ✏️ YOUR TURN: fill in the two ___ blanks below
    final = client.messages.create(
        model=MODEL,
        max_tokens=8000,
        system=SYSTEM_PROMPT,
        output_config={"format": RESOLUTION_SCHEMA},   # <-- pass RESOLUTION_SCHEMA here using {"format": RESOLUTION_SCHEMA}
        tool_choice={"type": "none"},     # <-- prevent further tool calls: {"type": "none"}
        thinking={"type": "adaptive"},
        messages=messages
    )
    return get_structured_result(final)


# result = run_agent_structured("Resolve ticket TKT-1042")
# print(json.dumps(result, indent=2))

---
### ✅ CHECKPOINT 1 — Working support ticket agent with structured output

You should have an agent that calls 2–3 tools per ticket and returns structured JSON.

**Verify:** `run_agent_structured("Resolve ticket TKT-1044")` → category: feature_request, suggests API workaround

---
# Part 2: Adaptive Thinking

The basic agent works, but TechFlow's support lead raises a concern: "Some tickets are straightforward — duplicate charges, password resets. Others are genuinely ambiguous. We want deeper reasoning on hard tickets without slowing down the easy ones. And when the agent gets a hard ticket wrong, we can't see *why* it made that call."

Adaptive thinking solves both problems. The `output_config.effort` parameter lets you control how deeply Claude reasons — "high" for complex or ambiguous tickets, "low" for straightforward ones. Thinking traces give you visibility into the decision process, so you can audit *why* the agent chose to escalate or which KB article it weighed most heavily.

> 📖 **Reference:** [Adaptive thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

## Cell 5: Add Effort-Level Thinking Control

Add `output_config.effort` to control how deeply Claude reasons. With high effort, Claude produces detailed thinking traces between tool calls — reasoning about what to search for, evaluating KB results, deciding whether to escalate. With low effort, it keeps reasoning brief for simple tickets. Display thinking blocks so you can see the agent's decision process.

### ✏️ YOUR TURN — implement `run_agent_thinking()`

Write the whole function this time — the comments walk you through the four steps.

In [6]:
# TODO: Add effort-level thinking control to the agent
# 1. Copy run_agent — run the tool loop with thinking={"type": "adaptive"}
#    and output_config={"effort": effort} (but NOT format — save that for the final call)
# 2. In the loop, display thinking blocks: block.type == "thinking"
# 3. After the tool loop ends, make a FINAL call with:
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
#    - Append a user message like "Provide your structured resolution as JSON."
# 4. Use get_structured_result() to parse the final response

def run_agent_thinking(user_message: str, effort: str = "high") -> dict:
    """Run agent with effort-controlled adaptive thinking."""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
        tools=tools, thinking={"type": "adaptive"},
        output_config={"effort": effort},
        messages=messages
    )

    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "thinking":
                print(f"\n[thinking] {block.thinking}\n")
            elif block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

        response = client.messages.create(
            model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
            tools=tools, thinking={"type": "adaptive"},
            output_config={"effort": effort},
            messages=messages
        )

    # Final call: constrain to structured JSON, no more tools
    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})
    final = client.messages.create(
        model=MODEL, max_tokens=8000, system=SYSTEM_PROMPT,
        output_config={"effort": effort, "format": RESOLUTION_SCHEMA},
        tool_choice={"type": "none"},
        thinking={"type": "adaptive"},
        messages=messages
    )
    return get_structured_result(final)

## Cell 6: Explore Adaptive Thinking in Action

MedTech Solutions (TKT-1046) is reporting intermittent 500 errors — 15% of API calls failing, no changes on their end, started at 2am. Is it rate limiting? A server-side outage? Network issues? The KB won't have a perfect match.

This is where thinking earns its keep. Run the ambiguous ticket with high effort, then compare it with low effort on the same ticket. Watch how reasoning depth changes — and ask yourself: for TechFlow's 500 tickets/day, how would you route simple vs. complex tickets to different effort levels?

In [7]:
def _not_built_yet(fn_name, result):
    """Build-along guard: the YOUR TURN stubs return None until you implement them."""
    if result is not None:
        return False
    print(f"\n[--] {fn_name}() isn't built yet - that's the exercise, not a bug.")
    print(f"     Find the '# YOUR TURN' marker inside {fn_name}(), build it, then run this again.")
    return True

# Run the ambiguous ticket at high effort — observe the thinking traces
print("=== TKT-1046: Intermittent API Errors (ambiguous) ===\n")
result = run_agent_thinking("Resolve ticket TKT-1046", effort="high")
if _not_built_yet("run_agent_thinking", result):
    raise SystemExit(0)
print(f"\nResolution:")
print(json.dumps(result, indent=2))

# Now compare: same ticket, low effort
print(f"\n\n{'='*50}")
print("=== Same ticket, LOW effort ===")
print(f"{'='*50}\n")

for effort in ["high", "low"]:
    start = time.time()
    result = run_agent_thinking("Resolve ticket TKT-1046", effort=effort)
    elapsed = time.time() - start
    print(f"\n[effort={effort}] Confidence: {result['confidence']} | Steps: {len(result['solution_steps'])} | Escalate: {result['escalation_needed']} | Time: {elapsed:.1f}s")

=== TKT-1046: Intermittent API Errors (ambiguous) ===




[thinking] 




[thinking] 




[thinking] 




Resolution:
{
  "diagnosis": "Customer (MedTech Solutions) is experiencing a sustained ~15% intermittent 500 error rate on production API calls beginning around 2am, with no changes made on their end. Retries often succeed, suggesting a server-side or infrastructure-level issue (e.g., load balancer misrouting, backend instability, or rate-limit responses being masked as 500s) rather than a client-side integration problem. Per KB-007, an error rate exceeding 10% for more than 1 hour meets the threshold for engineering escalation.",
  "solution_steps": [
    "Checked status page for known platform incidents correlating with the ~2am start time",
    "Advised customer to inspect X-RateLimit-* headers to rule out 429s being masked as 500 errors behind the load balancer",
    "Requested correlation data on whether failures cluster around specific endpoints or payload sizes",
    "Instructed customer to enable request ID logging and share failing request IDs/timestamps for server-side traci

...


[thinking] 




[effort=high] Confidence: medium | Steps: 7 | Escalate: True | Time: 28.5s



[thinking] 




[thinking] 




[effort=low] Confidence: medium | Steps: 7 | Escalate: True | Time: 25.6s


---
### ✅ CHECKPOINT 2 — Agent with reasoning visibility + effort control

Effort comparison should show observable differences in reasoning depth and quality.

---
# Part 3: Streaming

TechFlow's support agents use a real-time dashboard to monitor the AI triage system. When a ticket comes in, they need to *see* the agent working — which ticket it's looking up, what it's searching for, how it's reasoning. A 15-second spinner followed by a wall of text doesn't build trust.

Streaming solves this. Replace `create()` with `stream()` and tokens arrive in real-time: thinking traces flow as the agent reasons, tool calls appear as they're made, and the structured resolution streams at the end.

> 📖 **Reference:** [Streaming messages](https://docs.anthropic.com/en/docs/build-with-claude/streaming)

## Cell 7: Streaming Agentic Loop

Replace `client.messages.create()` with `client.messages.stream()` so TechFlow's dashboard can display the agent's work in real-time. Handle the key event types: `thinking_delta` (reasoning tokens), `text_delta` (final response), and `input_json_delta` (tool arguments).

Use `stream.get_final_message()` after the stream completes to get the full response object for loop continuation.

### ✏️ YOUR TURN — implement `run_agent_streaming()`

Write the whole function — swap `create()` for `stream()` and handle the stream events listed in the comments.

In [8]:
# TODO: Build the streaming agentic loop
# 1. Replace create() with stream() using a context manager (with ... as stream:)
#    Use output_config={"effort": effort} during the tool loop (NO format constraint)
# 2. Iterate over stream events, handling:
#    - content_block_start: check content_block.type (thinking/tool_use/text)
#    - content_block_delta: handle thinking_delta, text_delta, input_json_delta
# 3. After streaming, use stream.get_final_message() for the complete response
# 4. If stop_reason is tool_use, execute tools and continue the loop
# 5. After the tool loop ends, make a FINAL streamed call with:
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
# 6. Use get_structured_result() for the final JSON
# Remember: pass thinking={"type": "adaptive", "display": "summarized"} to stream()

def run_agent_streaming(user_message: str, effort: str = "high") -> dict:
    """Run agent with streaming output."""
    messages = [{"role": "user", "content": user_message}]

    def _stream_once(output_config, max_tokens=32000, tool_choice=None):
        kwargs = dict(
            model=MODEL,
            max_tokens=max_tokens,
            system=SYSTEM_PROMPT,
            tools=tools,
            thinking={"type": "adaptive", "display": "summarized"},
            output_config=output_config,
            messages=messages
        )
        if tool_choice is not None:
            kwargs["tool_choice"] = tool_choice

        with client.messages.stream(**kwargs) as stream:
            for event in stream:
                if event.type == "content_block_start":
                    if event.content_block.type == "thinking":
                        print("\n[thinking] ", end="", flush=True)
                    elif event.content_block.type == "tool_use":
                        print(f"\n[tool_use: {event.content_block.name}] ", end="", flush=True)
                    elif event.content_block.type == "text":
                        print("\n[text] ", end="", flush=True)
                elif event.type == "content_block_delta":
                    delta = event.delta
                    if delta.type == "thinking_delta":
                        print(delta.thinking, end="", flush=True)
                    elif delta.type == "text_delta":
                        print(delta.text, end="", flush=True)
                    elif delta.type == "input_json_delta":
                        print(delta.partial_json, end="", flush=True)
            return stream.get_final_message()

    response = _stream_once(output_config={"effort": effort})

    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

        response = _stream_once(output_config={"effort": effort})

    # Final streamed call: constrain to structured JSON, no more tools
    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})
    final = _stream_once(
        output_config={"effort": effort, "format": RESOLUTION_SCHEMA},
        max_tokens=8000,
        tool_choice={"type": "none"}
    )
    return get_structured_result(final)

## Cell 8: Full Demo

SecureNet Systems just submitted a critical ticket — their admin account is locked out and 47 team members are blocked. Run the full agent with streaming to see all four features composing in real-time: the agentic loop orchestrates tool calls, structured output guarantees the resolution format, adaptive thinking reasons through the security implications, and streaming shows it all as it happens.

In [9]:
print("Full Agent Demo: Resolving TKT-1045 (account lockout)")
print("   Streaming + Adaptive Thinking + Tools + Structured Output")
print("=" * 60)

start = time.time()
result = run_agent_streaming("Resolve ticket TKT-1045")
elapsed = time.time() - start
if _not_built_yet("run_agent_streaming", result):
    raise SystemExit(0)

print(f"\n\n{'=' * 60}")
print(f"Total time: {elapsed:.1f}s")
print(f"\nStructured Resolution:")
print(json.dumps(result, indent=2))

Full Agent Demo: Resolving TKT-1045 (account lockout)
   Streaming + Adaptive Thinking + Tools + Structured Output



[thinking] 

I

 should

 ret

...

"

confidence":"high","esc

alation_needed":true,

"category":"account"}



Total time: 25.9s

Structured Resolution:
{
  "diagnosis": "SecureNet Systems' admin account (admin@securenet.io) became locked after repeated failed MFA attempts. Because the organization's SSO/SAML authentication is routed through this single admin account (an Enterprise-tier feature), the lockout cascaded to block all 47 team members from platform access.",
  "solution_steps": [
    "Verify the lockout is due to failed MFA attempts and not a security compromise",
    "Initiate identity verification via the secondary recovery email on file for admin@securenet.io before any credential reset",
    "Enable temporary direct login bypass at /login/direct for the 47 affected non-admin users so they can resume work immediately",
    "Escalate to Account Security/Tier 2 team to perform the admin MFA reset via /admin/recover, which requires support-level override with manager approval",
    "Restore full admin/SSO access once identity is verified and MFA is reset (target within 1 hour given

---
### ✅ CHECKPOINT 3 — Real-time agent

All four features composing in a single agent run:
agentic loop + structured output + adaptive thinking + streaming.

---
# Extra Credit

1. **Tool choice controls** — Use `tool_choice: {"type": "none"}` to force Claude to stop calling tools
2. **Effort optimization** — Process all tickets at `effort="high"`, `effort="medium"`, `effort="low"`. Build a quality/speed table.
3. **Batch processing** — Use the Batch API to process all tickets at once (50% cost reduction).

## Learn More

- [Claude API Documentation](https://docs.anthropic.com/en/docs)
- [Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Adaptive Thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)
- [Structured Outputs](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)
- [Streaming](https://docs.anthropic.com/en/docs/build-with-claude/streaming)